# Building Machine Learning Optimization Engines from Scratch
By: Omosaiye Nehemiah Bukola

In this notebook, I build the core mathematical engines of Machine Learning (Gradient Descent, Sigmoid Activation, Log Loss, and Mini-Batch processing) entirely from scratch using pure Python and NumPy. No black-box libraries like scikit-learn are used. 

## Part 1: The Core Calculus of Gradient Descent
Before applying optimization to real data, we must prove the engine works on a pure mathematical function. Here, I built an engine to find the global minimum of the cost function $J(w) = w^4 - 4w^2 + 5$.

In [1]:
def cost_function(w):
    return (w**4) - (4 * (w**2)) + 5

def get_gradient(w):
    return (4 * w**3) - (8 * w)

In [2]:
def run_gradient_descent(w_init, alpha, epochs):
    w = w_init
    
    print(f"Initial State: w = {w:.4f}, Cost = {cost_function(w):.4f}")
    
    for i in range(epochs):
        # 1. Calculate the slope at our current position
        grad = get_gradient(w)
        
        # 2. Update the weight by stepping down the gradient
        w = w - (alpha * grad)
        
        current_cost = cost_function(w)
        
        print(f"Epoch {i+1}: w = {w:.4f}, Cost = {current_cost:.4f}")
            
    return w

In [3]:
final_w = run_gradient_descent(w_init=1.0, alpha=0.1, epochs=10)

Initial State: w = 1.0000, Cost = 2.0000
Epoch 1: w = 1.4000, Cost = 1.0016
Epoch 2: w = 1.4224, Cost = 1.0005
Epoch 3: w = 1.4092, Cost = 1.0002
Epoch 4: w = 1.4172, Cost = 1.0001
Epoch 5: w = 1.4124, Cost = 1.0000
Epoch 6: w = 1.4153, Cost = 1.0000
Epoch 7: w = 1.4136, Cost = 1.0000
Epoch 8: w = 1.4146, Cost = 1.0000
Epoch 9: w = 1.4140, Cost = 1.0000
Epoch 10: w = 1.4144, Cost = 1.0000


In [4]:
final_w = run_gradient_descent(w_init=1.0, alpha=0.8, epochs=10)

Initial State: w = 1.0000, Cost = 2.0000
Epoch 1: w = 4.2000, Cost = 245.6096
Epoch 2: w = -206.0016, Cost = 1800700302.6374
Epoch 3: w = 27972938.6102, Cost = 612283236091610237125278040064.0000
Epoch 4: w = -70042921939560992604160.0000, Cost = 24068943086633149778390066248475809263298365629051407646318777954083056636041689183501156352.0000
Epoch 5: w = 1099620286310814364648780761034106602286675915961506334243119974842368.0000, Cost = 1462079450842751551113360110993426740653736035280218833192960585184965322913386878305677334016005729312523293222246752540453020844834420786754634840910585465101328162878096780941155791421695182326005723472262433932351680402023507012095471585853421340565053342240943755078139904.0000


OverflowError: (34, 'Result too large')

### The Danger of a High Learning Rate
In the cell above, testing a learning rate of `0.8` caused the gradients to explode. The engine overshot the bottom of the valley and bounced up to infinity (`OverflowError`). This proves why tuning the learning rate ($\alpha$) is crucial in machine learning.

## Part 2: Linear Regression and 2D Gradient Descent
Next, we upgrade the engine to handle actual datasets with inputs ($X$) and outputs ($Y$). Here, we build an engine that calculates Mean Squared Error (MSE) and updates two variables simultaneously: the weight ($w$) and the bias ($b$).

In [5]:
import numpy as np

X = np.array([1.0, 2.0])
Y = np.array([2.0, 4.0])

def mse_cost(X, Y, w, b):
    predictions = w*X + b

    cost = np.mean((predictions - Y)**2)
    return cost

def get_gradients(X, Y, w, b):
    N = len(X)
    predictions = w*X + b

    dj_dw = (2/N) * np.sum(X * (predictions - Y))
    dj_db = (2/N) * np.sum(predictions - Y)

    return dj_dw, dj_db

In [6]:
gradients = get_gradients(X, Y, w=0.0, b=0.0)
print(f"dj_dw: {gradients[0]}, dj_db: {gradients[1]}")

dj_dw: -10.0, dj_db: -6.0


In [7]:
def run_2d_gradient_descent(X, Y, w_init, b_init, alpha, epochs):
    w = w_init
    b = b_init

    for i in range(epochs):
        dj_dw, dj_db = get_gradients(X, Y, w, b)

        w = w - alpha * dj_dw
        b = b - alpha * dj_db

        if i % 10 == 0:
            cost = mse_cost(X, Y, w, b)
            print(f"Epoch {i}: w = {w:.4f}, b = {b:.4f}, Cost = {cost:.4f}")

    return w, b

In [8]:
final_w, final_b = run_2d_gradient_descent(X, Y, w_init = 0.0, b_init = 0.0, alpha = 0.1, epochs = 100)

Epoch 0: w = 1.0000, b = 0.6000, Cost = 1.0600
Epoch 10: w = 1.5297, b = 0.7609, Cost = 0.0584
Epoch 20: w = 1.5940, b = 0.6569, Cost = 0.0435
Epoch 30: w = 1.6495, b = 0.5671, Cost = 0.0324
Epoch 40: w = 1.6974, b = 0.4896, Cost = 0.0242
Epoch 50: w = 1.7388, b = 0.4227, Cost = 0.0180
Epoch 60: w = 1.7745, b = 0.3649, Cost = 0.0134
Epoch 70: w = 1.8053, b = 0.3150, Cost = 0.0100
Epoch 80: w = 1.8319, b = 0.2720, Cost = 0.0075
Epoch 90: w = 1.8549, b = 0.2348, Cost = 0.0056


### Upgrading to Stochastic Gradient Descent (SGD)
Standard Batch Gradient Descent (above) calculates the error for the *entire* dataset before taking a single step. For massive real-world datasets, this is too slow. 
Below, I implemented **Stochastic Gradient Descent (SGD)**, which updates the weights after looking at *every single data point*, one by one. It is much faster, though slightly more chaotic.

In [9]:
def run_stochastic_gradient_descent(X, Y, w_init, b_init, alpha, epochs):
    w = w_init
    b = b_init

    for epoch in range(epochs):

        for i in range(len(X)):

            x_i = X[i]
            y_i = Y[i]

            prediction = w*x_i + b

            dj_dw = 2*x_i*(w*x_i + b - y_i)
            dj_db = 2*(w*x_i + b - y_i)

            w = w - alpha*dj_dw
            b = b - alpha*dj_db

        if epoch % 10 == 0:
            print(f"Epoch {epoch}: w ={w:.4f}, b = {b:.4f}")

    return w, b
        

In [10]:
final_w, final_b = run_stochastic_gradient_descent(X, Y, w_init = 0.0, b_init=0.0, alpha=0.1, epochs=100) 

Epoch 0: w =1.5200, b = 0.9600
Epoch 10: w =1.6809, b = 0.6382
Epoch 20: w =1.7878, b = 0.4243
Epoch 30: w =1.8589, b = 0.2821
Epoch 40: w =1.9062, b = 0.1876
Epoch 50: w =1.9377, b = 0.1247
Epoch 60: w =1.9586, b = 0.0829
Epoch 70: w =1.9724, b = 0.0551
Epoch 80: w =1.9817, b = 0.0366
Epoch 90: w =1.9878, b = 0.0244


### The Industry Standard: Mini-Batch Gradient Descent
To get the best of both worlds (the stability of Batch and the speed of SGD), modern AI uses **Mini-Batch Gradient Descent**. 
Below, I wrote the algorithm to slice the dataset into smaller batches, average the gradients for just that batch, and update the weights.

In [11]:
def run_minibatch_gradient_descent(X, Y, w_init, b_init, alpha, epochs, batch_size):
    w = w_init
    b = b_init
    N = len(X)

    for epoch in range(epochs):

        for i in range(0, N, batch_size):

            X_batch = X[i : i + batch_size]
            Y_batch = Y[i : i + batch_size]
            m = len(X_batch)

            predictions = w*X_batch + b

            dj_dw = 2/m * np.sum(X_batch*(predictions - Y_batch))
            dj_db = 2/m * np.sum(predictions - Y_batch)

            w = w - (alpha * dj_dw)
            b = b - (alpha * dj_db)

        if epoch % 10 == 0:
            print(f"Epoch {epoch}: w = {w:.4f}, b = {b:.4f}")

    return w, b

In [12]:
# Our updated 4-point dataset
X_large = np.array([1.0, 2.0, 3.0, 4.0])
Y_large = np.array([2.0, 4.0, 6.0, 8.0])

final_w, final_b = run_minibatch_gradient_descent(X_large, Y_large, w_init=0.0, b_init=0.0, alpha=0.01, epochs=100, batch_size=2)

Epoch 0: w = 0.5708, b = 0.1918
Epoch 10: w = 1.7845, b = 0.5675
Epoch 20: w = 1.8225, b = 0.5418
Epoch 30: w = 1.8338, b = 0.5093
Epoch 40: w = 1.8439, b = 0.4785
Epoch 50: w = 1.8533, b = 0.4497
Epoch 60: w = 1.8621, b = 0.4225
Epoch 70: w = 1.8705, b = 0.3970
Epoch 80: w = 1.8783, b = 0.3730
Epoch 90: w = 1.8856, b = 0.3505


## Part 3: Classification and the Artificial Neuron
Linear regression predicts continuous numbers, but for classification (e.g., Cat vs. Dog, Benign vs. Malignant), we need probabilities between 0 and 1. 

By wrapping our linear equation ($wX + b$) inside an **Activation Function**, we create an Artificial Neuron. Below is the mathematics for the **Sigmoid Function**, which squishes any real number into a percentage probability.

In [13]:
import numpy as np

def sigmoid(z):
    value = 1/(1 + np.exp(-1*z))
    return value

In [14]:
test_z = np.array([0, 5, -5])
print(sigmoid(test_z))

[0.5        0.99330715 0.00669285]


### Binary Cross-Entropy (Log Loss)
Because the Sigmoid function warps our predictions, Mean Squared Error (MSE) no longer works reliably (it creates non-convex, bumpy loss landscapes). 
Instead, we must build a new ruler: **Log Loss**. This function heavily penalizes the model using logarithms if it is extremely confident but completely wrong.

In [15]:
def log_loss(Y_true, Y_pred):
    loss = np.mean(-(Y_true*np.log(Y_pred) + (1-Y_true)*np.log(1-Y_pred)))
    return loss
    

In [16]:
Y_test = np.array([1.0, 1.0])
Y_guesses = np.array([0.9, 0.01])

print(log_loss(Y_test, Y_guesses))

2.3552653508229584


## Part 4: Building a Logistic Regression Neural Network
We now combine the Optimizer (Gradient Descent), the Predictor (Sigmoid), and the Ruler (Log Loss) to solve a real-world problem. 
Using a 1D slice of the Wisconsin Breast Cancer dataset, this engine will learn to predict the probability that a tumor is malignant based solely on its size.

In [17]:
X_tumor = np.array([0.8, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0, 2.2, 2.4, 2.6])
Y_malignant = np.array([0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0])

def run_logistic_regression(X, Y, w_init, b_init, alpha, epochs):
    w = w_init
    b = b_init

    for epoch in range(epochs):

        for i in range(len(X)):
            x_i = X[i]
            y_i = Y[i]

            prediction = sigmoid(w*x_i + b)

            dj_dw = 2 * x_i * (prediction - y_i)
            dj_db = 2 * (prediction - y_i)

            w = w - (alpha*dj_dw)
            b = b - (alpha*dj_db)

        if epoch % 1000 == 0:

            current_predictions = sigmoid(w * X + b)
            cost = log_loss(Y, current_predictions)
            print(f"Epoch {epoch}: w = {w:.4f}, b = {b:.4f}, Loss = {cost:.4f}")
    return w, b
            

In [18]:
final_w, final_b = run_logistic_regression(X_tumor, Y_malignant, w_init=0.0, b_init=0.0, alpha=0.1, epochs=10000)

Epoch 0: w = 0.5718, b = 0.0950, Loss = 0.6965
Epoch 1000: w = 11.1771, b = -18.8085, Loss = 0.0651
Epoch 2000: w = 14.3361, b = -24.2221, Loss = 0.0460
Epoch 3000: w = 16.5721, b = -28.0451, Loss = 0.0366
Epoch 4000: w = 18.3508, b = -31.0825, Loss = 0.0306
Epoch 5000: w = 19.8425, b = -33.6282, Loss = 0.0264
Epoch 6000: w = 21.1332, b = -35.8295, Loss = 0.0233
Epoch 7000: w = 22.2732, b = -37.7733, Loss = 0.0208
Epoch 8000: w = 23.2955, b = -39.5158, Loss = 0.0188
Epoch 9000: w = 24.2228, b = -41.0960, Loss = 0.0172


In [19]:
X_tumor = np.array([0.8, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0, 2.2, 2.4, 2.6])
Y_malignant = np.array([0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0])

def run_logistic_regression_mb(X, Y, w_init, b_init, alpha, epochs, batch_size):
    w = w_init
    b = b_init
    N = len(X)
    
    for epoch in range(epochs):

        for i in range(0, N, batch_size):
            x_batch = X[i : i + batch_size]
            y_batch = Y[i : i + batch_size]
            m = len(x_batch)
            prediction = sigmoid(w*x_batch + b)

            dj_dw = 2/m* np.sum(x_batch * (prediction - y_batch))
            dj_db = 2 /m * np.sum(prediction - y_batch)

            w = w - (alpha*dj_dw)
            b = b - (alpha*dj_db)

        if epoch % 1000 == 0:

            current_predictions = sigmoid(w * X + b)
            cost = log_loss(Y, current_predictions)
            print(f"Epoch {epoch}: w = {w:.4f}, b = {b:.4f}, Loss = {cost:.4f}")
    return w, b
            

In [20]:
final_w, final_b = run_logistic_regression_mb(X_tumor, Y_malignant, w_init=0.0, b_init=0.0, alpha=0.1, epochs=10000, batch_size=2)

Epoch 0: w = 0.3296, b = 0.0491, Loss = 0.6606
Epoch 1000: w = 8.5636, b = -14.4383, Loss = 0.0893
Epoch 2000: w = 10.9847, b = -18.5863, Loss = 0.0659
Epoch 3000: w = 12.6970, b = -21.5105, Loss = 0.0544
Epoch 4000: w = 14.0729, b = -23.8567, Loss = 0.0470
Epoch 5000: w = 15.2427, b = -25.8499, Loss = 0.0416
Epoch 6000: w = 16.2696, b = -27.5987, Loss = 0.0375
Epoch 7000: w = 17.1896, b = -29.1650, Loss = 0.0342
Epoch 8000: w = 18.0258, b = -30.5883, Loss = 0.0315
Epoch 9000: w = 18.7940, b = -31.8956, Loss = 0.0292


### Inference: Diagnosing an Unseen Patient
Training the model is only half the job. Our engine found the mathematical decision boundary separating safe tumors from dangerous ones to be roughly **1.697 cm**. 

Below, we pass a completely unseen tumor size (**1.75 cm**) to our trained model to see its diagnosis. Because it is slightly larger than the boundary, the AI correctly identifies it as leaning malignant, but quantifies its uncertainty mathematically.

In [21]:
new_tumor = 1.75
# Step 1: Calculate the linear z value
z = final_w * new_tumor + final_b

# Step 2: Squish it into a probability
probability = sigmoid(z)

print(f"The AI predicts a {probability * 100:.2f}% chance of malignancy.")

The AI predicts a 73.66% chance of malignancy.
